### 1. schema.py


In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ExtractedCV(BaseModel):
    name: str
    contact_info: str
    core_skills: List[str]
    projects: List[dict] = Field(description="List of dicts: 'name' and 'expanded_bullets'")
    experience: List[dict] = Field(description="List of dicts: 'role', 'company', 'dates', 'expanded_bullets'")
    education: List[dict] = Field(description="List of dicts: 'degree', 'university', 'dates'")

class JobRequirements(BaseModel):
    job_title: str
    mandatory_skills: List[str]
    preferred_skills: List[str]
    core_responsibilities: List[str]
    company_tone: str

class TailoredDraft(BaseModel):
    name: str
    contact_info: str
    summary: str
    selected_skills: List[str]
    tailored_projects: List[dict]
    tailored_experience: List[dict]
    education: List[dict]

class SupervisorReview(BaseModel):
    is_approved: bool
    feedback: str

### 2. agents.py

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from schemas import ExtractedCV, JobRequirements, TailoredDraft, SupervisorReview

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("⚠️ GEMINI_API_KEY not found in .env file.")

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-2.5-flash"

def run_profiler_agent(raw_cv_text: str) -> ExtractedCV:
    print("🕵️‍♂️ Profiler Agent: Analyzing raw CV...")
    prompt = f"Extract the details from this CV and expand every project and job into detailed bullet points focusing on tools and impact.\n\nRaw CV:\n{raw_cv_text}"
    response = client.models.generate_content(
        model=MODEL_NAME, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=ExtractedCV, temperature=0.1)
    )
    return ExtractedCV.model_validate_json(response.text)

def run_recruiter_agent(raw_jd_text: str) -> JobRequirements:
    print("👔 Recruiter Agent: Extracting job requirements...")
    prompt = f"Analyze this job description. Extract the job title, mandatory/preferred skills, core responsibilities, and tone.\n\nJob Description:\n{raw_jd_text}"
    response = client.models.generate_content(
        model=MODEL_NAME, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=JobRequirements, temperature=0.1)
    )
    return JobRequirements.model_validate_json(response.text)

def run_tailor_agent(cv_data: ExtractedCV, jd_data: JobRequirements, feedback: str = "") -> TailoredDraft:
    print("✍️ Tailor Agent: Drafting customized CV...")
    feedback_section = f"\nSUPERVISOR FEEDBACK TO FIX:\n{feedback}" if feedback else ""
    prompt = f"""
    You are an elite Career Strategist drafting a tailored CV.
    RULES:
    1. ZERO HALLUCINATIONS. Only use facts from the Master Profile.
    2. Choose only the most relevant projects for this specific job.
    3. Select exactly 3-4 bullet points per project/role that best match the Job Requirements.
    4. Write a new 3-sentence summary hooking this specific employer.
    5. IDENTITY: Copy the user's exact 'name', 'contact_info', and 'education' from the Master Profile directly into your output without changing them.
    {feedback_section}
    
    Master Profile: {cv_data.model_dump_json()}
    Job Requirements: {jd_data.model_dump_json()}
    """
    response = client.models.generate_content(
        model=MODEL_NAME, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=TailoredDraft, temperature=0.3)
    )
    return TailoredDraft.model_validate_json(response.text)

def run_supervisor_agent(draft: TailoredDraft, jd_data: JobRequirements) -> SupervisorReview:
    print("🧐 Supervisor Agent: Auditing the draft...")
    prompt = f"""
    Review the proposed CV Draft against the Job Requirements.
    If it is perfect, set 'is_approved' to True and leave feedback empty.
    If it is flawed, set 'is_approved' to False and write strict feedback on what to fix.
    
    CV Draft: {draft.model_dump_json()}
    Job Requirements: {jd_data.model_dump_json()}
    """
    response = client.models.generate_content(
        model=MODEL_NAME, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=SupervisorReview, temperature=0.1)
    )
    return SupervisorReview.model_validate_json(response.text)

### 3. main.py

In [ ]:
import os
import json
import PyPDF2

from agents import (
    run_profiler_agent, 
    run_recruiter_agent, 
    run_tailor_agent, 
    run_supervisor_agent
)

def extract_text_from_pdf(pdf_path: str) -> str:
    print(f"📄 Reading {pdf_path}...")
    text = ""
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text()
    return text

def safe_latex(text):
    """Escapes special LaTeX characters to prevent compile errors."""
    if not isinstance(text, str):
        return str(text)
    return text.replace("%", "\\%").replace("$", "\\$").replace("&", "\\&").replace("_", "\\_")

def generate_latex_from_file(draft_data, template_filename="template.tex", output_filename="Tailored_Application.tex"):
    print(f"🖨️ Publisher Agent: Reading '{template_filename}' and formatting document...")
    
    try:
        with open(template_filename, 'r', encoding='utf-8') as file:
            template_text = file.read()
    except FileNotFoundError:
        print(f"❌ ERROR: Could not find {template_filename}.")
        return

    skills_str = ", ".join(draft_data.selected_skills)
    
    projects_tex = ""
    for proj in draft_data.tailored_projects:
        projects_tex += f"\\noindent\\textbf{{{safe_latex(proj.get('name', 'Project'))}}} \\hfill \\textit{{Relevant Project}}\\\\\n"
        projects_tex += "\\begin{itemize}[leftmargin=0.15in, itemsep=-2pt]\n"
        for bullet in proj.get('expanded_bullets', proj.get('bullets', [])):
            projects_tex += f"    \\item {safe_latex(bullet)}\n"
        projects_tex += "\\end{itemize}\n\\vspace{4pt}\n"

    experience_tex = ""
    for job in draft_data.tailored_experience:
        company = safe_latex(job.get('company', ''))
        company_str = f" $|$ \\textit{{{company}}}" if company else ""
        experience_tex += f"\\noindent\\textbf{{{safe_latex(job.get('role', 'Role'))}}}{company_str} \\hfill {safe_latex(job.get('dates', 'Dates'))}\\\\\n"
        experience_tex += "\\begin{itemize}[leftmargin=0.15in, itemsep=-2pt]\n"
        for bullet in job.get('expanded_bullets', job.get('bullets', [])):
            experience_tex += f"    \\item {safe_latex(bullet)}\n"
        experience_tex += "\\end{itemize}\n\\vspace{4pt}\n"

    education_tex = ""
    for edu in draft_data.education:
        education_tex += f"\\noindent\\textbf{{{safe_latex(edu.get('university', 'University'))}}} \\hfill {safe_latex(edu.get('dates', 'Dates'))}\\\\\n"
        education_tex += f"\\textit{{{safe_latex(edu.get('degree', 'Degree'))}}}\\\\\n\\vspace{{2pt}}\n\n"

    # Inject the tags
    final_tex = template_text.replace("<<NAME>>", safe_latex(draft_data.name))
    final_tex = final_tex.replace("<<CONTACT>>", safe_latex(draft_data.contact_info))
    final_tex = final_tex.replace("<<SUMMARY>>", safe_latex(draft_data.summary))
    final_tex = final_tex.replace("<<SKILLS>>", safe_latex(skills_str))
    final_tex = final_tex.replace("<<PROJECTS>>", projects_tex)
    final_tex = final_tex.replace("<<EXPERIENCE>>", experience_tex)
    final_tex = final_tex.replace("<<EDUCATION>>", education_tex)

    with open(output_filename, 'w', encoding='utf-8') as file:
        file.write(final_tex)
    print(f"✅ Final Universal CV successfully generated: {output_filename}")

def main():
    print("🚀 INITIALIZING MULTI-AGENT CV PIPELINE...\n" + "="*40)
    
    cv_file = "CV.pdf"
    
    print("📋 Paste the target Job Description below.")
    print("(When finished, press Enter, type 'DONE', and press Enter again)\n")
    
    lines = []
    while True:
        line = input()
        if line.strip().upper() == 'DONE':
            break
        lines.append(line)
        
    job_description_text = "\n".join(lines)
    
    if not job_description_text.strip():
        print("⚠️ No job description provided. Exiting.")
        return
        
    raw_cv_text = extract_text_from_pdf(cv_file)
    profiler_data = run_profiler_agent(raw_cv_text)
    print("✅ Profiler Agent finished building Master Database.\n")
    
    recruiter_data = run_recruiter_agent(job_description_text)
    print("✅ Recruiter Agent finished extracting Job Requirements.\n")
    
    MAX_RETRIES = 2
    attempt = 1
    is_approved = False
    feedback = ""
    final_draft = None
    
    while attempt <= MAX_RETRIES and not is_approved:
        print(f"🔄 Routing to Tailor Agent (Attempt {attempt}/{MAX_RETRIES})...")
        draft = run_tailor_agent(profiler_data, recruiter_data, feedback)
        
        print("🕵️‍♂️ Routing to Supervisor Agent for QA Audit...")
        review = run_supervisor_agent(draft, recruiter_data)
        
        if review.is_approved:
            print("🟢 SUPERVISOR APPROVED: The draft meets all criteria!")
            is_approved = True
            final_draft = draft
        else:
            print(f"🔴 SUPERVISOR REJECTED. Feedback: {review.feedback}")
            feedback = review.feedback
            attempt += 1

    if not is_approved:
        print("⚠️ MAX RETRIES REACHED. Supervisor forced to approve the latest draft.")
        final_draft = draft
        
    # Save the raw JSON data
    json_filename = "Tailored_CV_Data.json"
    with open(json_filename, "w", encoding="utf-8") as f:
        json.dump(final_draft.model_dump(), f, indent=4, ensure_ascii=False)
        
    # Build the LaTeX file
    generate_latex_from_file(final_draft)
        
    print("="*40)
    print(f"🎉 PIPELINE COMPLETE! Your tailored CV is ready in 'Tailored_Application.tex'")

if __name__ == "__main__":
    main()